# Chimpanzee identification demo

This notebook lets you identify a chimpanzee from a photo of its face using
our fine-tuned ChimpUFE model. Everything is self-contained inside this
folder — no setup or coding knowledge required.

## How to use

1. **Put your photos** (cropped chimp faces, JPG or PNG) into the
   **`input/`** folder next to this notebook.
2. Click **Run All** in the toolbar (or press the ▶ button next to each cell
   from top to bottom).
3. The notebook will:
   - load the trained model from `model/best_model.pt`,
   - build a reference gallery from the example faces in `exemple/`
     (one folder per chimpanzee, named with its 2-letter code),
   - run a quick demo on one example per chimp,
   - then identify every image you placed in `input/`.

For each query image you will see the top-5 candidates with a confidence
score and a thumbnail of the closest gallery photo.

## Folder layout

```
demo/
├── image_inference.ipynb   ← this notebook
├── model/best_model.pt     ← trained weights
├── face_embedder/          ← model source code
├── exemple/<XX>/*.png      ← reference gallery (one folder per chimp)
└── input/                  ← drop your photos here
```

The 2-letter codes (`AD`, `BL`, `BS`, …) map to chimp names in the table
printed below.


In [1]:
# =============================================================================
# 1. Setup — install dependencies if missing, import utils, pick device.
# =============================================================================
import sys
from pathlib import Path

# Make the local face_embedder package and utils.py importable.
NOTEBOOK_DIR = Path.cwd().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import utils
utils.ensure_dependencies()

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Working folder: {NOTEBOOK_DIR}")


Using device: cuda
Working folder: /home/users/t/r/trixen/ChimpRec/Code/Recognition/ChimpUFE/demo


In [ ]:
# =============================================================================
# 2. Configuration — paths and identity table (see utils.py for defaults).
# =============================================================================
MODEL_PATH  = NOTEBOOK_DIR / "model" / "*.pt"
GALLERY_DIR = NOTEBOOK_DIR / "exemple"   # reference faces (one folder per chimp)
INPUT_DIR   = NOTEBOOK_DIR / "input"     # your query images go here
TOP_K       = utils.TOP_K
TEMPERATURE = utils.TEMPERATURE

INPUT_DIR.mkdir(exist_ok=True)

assert MODEL_PATH.exists(), (
    f"Model file not found: {MODEL_PATH}\n"
    f"Make sure model/best_model.pt is present next to this notebook."
)
assert GALLERY_DIR.exists() and any(GALLERY_DIR.iterdir()), (
    f"Reference gallery not found or empty: {GALLERY_DIR}"
)

print(f"Model file   : {MODEL_PATH}")
print(f"Gallery dir  : {GALLERY_DIR}")
print(f"Input dir    : {INPUT_DIR}")
print(f"Known chimps : {len(utils.INITIAL_TO_NAME)}")


Model file   : /home/users/t/r/trixen/ChimpRec/Code/Recognition/ChimpUFE/demo/model/best_model.pt
Gallery dir  : /home/users/t/r/trixen/ChimpRec/Code/Recognition/ChimpUFE/demo/exemple
Input dir    : /home/users/t/r/trixen/ChimpRec/Code/Recognition/ChimpUFE/demo/input
Known chimps : 20


## 3. Load the trained model
This cell loads the ChimpUFE backbone and the fine-tuned weights from
`model/best_model.pt`. The first run takes a few seconds.


In [3]:
# =============================================================================
# 3. Build the model (ViT-B/14 backbone + identity head) and load weights.
# =============================================================================
model, eval_tfm, class_to_idx, cfg = utils.load_model(MODEL_PATH, device)


Loading checkpoint ...
Loaded. classes=20  arcface=True  missing=0  unexpected=0


## 4. Build the reference gallery
We embed every image inside `exemple/<XX>/` once and average the embeddings
per chimp. Each query is then compared (cosine similarity) against these
20 prototypes to decide who it most resembles.


In [4]:
# =============================================================================
# 4. Embed the gallery (exemple/) and build per-chimp prototypes.
# =============================================================================
gallery = utils.build_gallery(GALLERY_DIR, model, eval_tfm, device)


  AD (Amadi      ) -> 14 images
  BL (Banalia    ) -> 14 images
  BS (Binasera   ) -> 11 images
  DK (Djiku      ) -> 13 images
  IV (Ivan       ) -> 13 images
  JJ (Jeje       ) -> 10 images
  KG (Kassongo   ) -> 8 images
  KM (Kalemi     ) -> 10 images
  KR (Kira       ) -> 11 images
  LM (Lwama      ) -> 13 images
  MG (Malago     ) -> 11 images
  MK (Muke       ) -> 11 images
  MM (Maniema    ) -> 12 images
  MZ (Mazingira  ) -> 11 images
  NJ (Nganja     ) -> 9 images
  NR (Nzuri      ) -> 12 images
  PD (Penda      ) -> 12 images
  TC (Tanganica  ) -> 11 images
  TS (Talisa     ) -> 11 images
  TT (Tingitingi ) -> 10 images

Gallery ready: 227 images across 20 chimps.


## 5. The `identify_chimp()` function
Pass any image path; it returns the top-5 candidates and (by default) shows
the query alongside the closest gallery photo of each candidate.


In [5]:
# =============================================================================
# 5. identify_chimp(image_path) -> top-K candidates + side-by-side plot.
# =============================================================================
def identify_chimp(image_path, top_k: int = TOP_K, show: bool = True):
    return utils.identify_chimp(
        image_path, gallery, model, eval_tfm, device,
        top_k=top_k, temperature=TEMPERATURE, show=show,
    )


print("Helper ready. Example usage:")
print("    identify_chimp('input/myphoto.jpg')")


Helper ready. Example usage:
    identify_chimp('input/myphoto.jpg')


## 7. Identify the chimps in your `input/` folder
Place one or more cropped chimp-face images inside `input/` and re-run this
cell. The notebook will display each query image with the top-5 candidates
and a summary table at the bottom.


In [1]:
                                        # =============================================================================
# 7. Run the identifier on every image inside the input/ folder.
# =============================================================================
import pandas as pd

query_paths = utils.list_images(INPUT_DIR)
print(f"Found {len(query_paths)} image(s) in: {INPUT_DIR}")

if not query_paths:
    print("\nNo images yet -> drop some .jpg/.png files into the 'input/' "
          "folder next to this notebook and re-run this cell.")
else:
    results = []
    for q in query_paths:
        print(f"\n=== {q.name} ===")
        df = identify_chimp(q, top_k=TOP_K, show=True)
        display(df.style.format({"cosine_sim": "{:.3f}",
                                 "confidence": "{:.1%}"}))
        results.append({
            "file":       q.name,
            "pred_code":  df["code"].iloc[0],
            "pred_name":  df["name"].iloc[0],
            "confidence": df["confidence"].iloc[0],
        })

    summary = pd.DataFrame(results)
    print("\n================  SUMMARY  ================")
    display(summary.style.format({"confidence": "{:.1%}"}))


NameError: name 'utils' is not defined